In [ ]:
# Fabric-native parameters and config
# Edit these values directly in Fabric, or pass them from a Fabric Data Pipeline.
ENV = "dev"
SCHEMA = "dbo"
MARKET = "AU"
MOCK_IPSTACK = True
LOOKBACK_DAYS = 90
LISTINGS_FILE = "listings.csv"
VIEWS_FILE = "property_views_5k.csv"

try:
    from notebookutils import mssparkutils
except ImportError:
    mssparkutils = None

from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

class Config:
    def __init__(self):
        self.env = ENV
        self.schema = SCHEMA
        self.market = MARKET
        self.mock_ipstack = MOCK_IPSTACK
        self.lookback_days = LOOKBACK_DAYS
        self.listings_file = LISTINGS_FILE
        self.views_file = VIEWS_FILE
        self.files_base = "Files"  # Fabric attached Lakehouse path
        self.listings_path = f"{self.files_base}/raw/sample/{self.listings_file}"
        self.views_path = f"{self.files_base}/raw/sample/{self.views_file}"
        self.fixture_base = f"{self.files_base}/fixtures/ipstack"

        self.bronze_listings = "bronze_listings"
        self.bronze_property_views = "bronze_property_views"
        self.bronze_ipstack_raw = "bronze_ipstack_raw"
        self.bronze_ipstack_errors = "bronze_ipstack_errors"
        self.silver_listings = "silver_listings"
        self.silver_ip_dim = "silver_ip_dim"
        self.silver_visits_enriched = "silver_visits_enriched"
        self.gold_suburb_interest = "gold_suburb_interest"
        self.gold_property_type_by_suburb = "gold_property_type_by_suburb"
        self.gold_price_engagement = "gold_price_engagement"
        self.gold_conversion_gaps = "gold_conversion_gaps"
        self.gold_property_trends = "gold_property_trends"
        self.gold_region_type_preference = "gold_region_type_preference"
        self.gold_repeat_interest = "gold_repeat_interest"
        self.gold_interstate_flow = "gold_interstate_flow"

    def table_fqn(self, table: str) -> str:
        return f"{self.schema}.{table}"

conf = Config()
print(f"env={conf.env} market={conf.market} mock={conf.mock_ipstack} views={conf.views_file}")
print(f"listings_path={conf.listings_path}")
print(f"views_path={conf.views_path}")



In [ ]:
from pyspark.sql import functions as F

class BronzeLoader:
    def __init__(self):
        self.c = conf

    def load_listings(self):
        df = (spark.read.option("header", True).csv(self.c.listings_path)
              .withColumn("price", F.col("price").cast("int"))
              .withColumn("bedrooms", F.col("bedrooms").cast("int"))
              .withColumn("bathrooms", F.col("bathrooms").cast("int"))
              .withColumn("land_size_sqm", F.col("land_size_sqm").cast("int"))
              .withColumn("is_luxury", F.col("is_luxury").cast("boolean"))
              .withColumn("_ingested_at", F.current_timestamp()))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.bronze_listings))
        print(f"✅ bronze_listings: {df.count()} rows")

    def load_views(self):
        cutoff = F.date_sub(F.current_date(), self.c.lookback_days)
        df = (spark.read.option("header", True).csv(self.c.views_path)
              .withColumn("event_ts", F.to_timestamp("event_ts"))
              .withColumn("view_duration_sec", F.col("view_duration_sec").cast("int"))
              .withColumn("enquiry_flag", F.col("enquiry_flag").cast("boolean"))
              .withColumn("favorite_flag", F.col("favorite_flag").cast("boolean"))
              .withColumn("event_date", F.to_date("event_ts"))
              .withColumn("_ingested_at", F.current_timestamp())
              .filter(F.col("event_date") >= cutoff))
        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(self.c.table_fqn(self.c.bronze_property_views))
        print(f"✅ bronze_property_views: {df.count()} rows appended")

    def run(self):
        self.load_listings()
        self.load_views()

BronzeLoader().run()


